In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5677.65it/s]


In [37]:
print("Model: \n", model,'\n')
print("Model Type: \n", type(model), '\n')
print("Model Config: \n", model.config,'\n', model.config.model_type, '\n')

Model: 
 LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-0

In [7]:
total_params = sum(p.numel() for p in model.parameters())
print("Total Model Parameters: ", round(total_params/1e6,2),'M')

Total Model Parameters:  134.52 M


In [9]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total Trainable Model Parameters: ", round(total_params/1e6,2),'M')
print("Trainable Percentage(%)", (trainable_params/total_params)*100)

Total Trainable Model Parameters:  134.52 M
Trainable Percentage(%) 100.0


In [10]:
print(tokenizer)

GPT2Tokenizer(name_or_path='HuggingFaceTB/SmolLM2-135M-Instruct', vocab_size=49152, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<filename>", rstr

In [38]:
print("Vocabulary size:", tokenizer.vocab_size)
print("BOS token: ", tokenizer.bos_token)
print("BOS ID: ", tokenizer.bos_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)
print("PAD ID:", tokenizer.pad_token_id)

Vocabulary size: 49152
BOS token:  <|im_start|>
BOS ID:  1
EOS token: <|im_end|>
EOS ID: 2
PAD token: <|im_end|>
PAD ID: 2


In [12]:
# Checking models native conversational formatting
print(tokenizer.chat_template)

{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [31]:
messages = [
    {
        "role":'user',
        "content":'I forgot my password and can not access my account.'
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(prompt)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
I forgot my password and can not access my account.<|im_end|>
<|im_start|>assistant



In [32]:
# text -> tokens
inputs = tokenizer(
    prompt,
    return_tensors='pt'
)
print("Tokenized input tensors:\n",inputs,'\n')
print("Input ids:\n",inputs['input_ids'],'\n')
print("Input ids shape:\n",inputs['input_ids'].shape,'\n')

Tokenized input tensors:
 {'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198,    57, 28349,   957,  8824,   284,   416,
           441,  1594,   957,  2051,    30,     2,   198,     1,   520,  9531,
           198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])} 

Input ids:
 tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198,    57, 28349,   957,  8824,   284,   416,
           441,  1594,   957,  2051,    30,     2,   198,     1,   520,  9531,
           198]]) 

Input ids shape:
 torch.Size([1, 41]) 



In [33]:
# decoding the tensor
decoded = tokenizer.decode(
    inputs['input_ids'][0]
)
print(decoded)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
I forgot my password and can not access my account.<|im_end|>
<|im_start|>assistant



In [34]:
# Generating from base model
import torch

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt=True,
    return_tensors='pt'
)


In [ ]:
with torch.no_grad():
    outputs = model.generate(
        inputs['input_ids'],
        max_new_tokens=200
    )
print("Raw Output: ", outputs) 

Raw Output:  tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198,    57, 28349,   957,  8824,   284,   416,
           441,  1594,   957,  2051,    30,     2,   198,     1,   520,  9531,
           198,    57,  5248, 22657,   327,   260,  9563,    30,   339,  5248,
          1535,   288,  4237,   346,   351,   750,  1974,   346,  1124,   325,
          6371,    30, 16222,   346,  5361,  2505,   549,   732,   506,  5024,
            47,  1431,   665,  1488,  1678,   338,   506,  4439,   346,   288,
          6616,   469,  8824,    47,     2]])


In [36]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)
print(response)

system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
I forgot my password and can not access my account.
assistant
I'm sorry for the confusion. I'm here to assist you with any issues you might be facing. Could you please tell me what's happened? Is there something specific that's causing you to forget your password?


### Baseline Model Response:
Before generating thousands of examples, let's formalize the task.
The baseline model response is generated. We will keep it mind, also 
make sure to remember that out desired output format is

INTENT : one allowed intent

PRIORITY : <HIGH | MEDIUM | LOW>

ACTION : one aciton suggested

INPUT : Natural-language technical support request

EXAMPLE : Someone got access to my account and changed the email address.


OUTPUT 

INTENT: allowed intent

PRIORITY: <low|medium|high>

ACTION: one concise action 

ALLOWED_INTENTS = {
    "password_reset",
    "billing_issue",
    "account_compromise",
    "refund_request",
    "technical_problem",
    "feature_request",
    "account_closure",
}

ALLOWED_PRIORITIES = {
    "low",
    "medium",
    "high",
}


For our Project
| Layer              | Example                                                           |
| ------------------ | ----------------------------------------------------------------- |
| Training objective | SFT language-model loss                                           |
| Task objective     | Correctly classify support requests + produce structured response |
| Evaluation         | Intent accuracy, priority accuracy, format validity, exact match  |


This distinction will matter enormously when we start interpreting experiments.
